#Read csv file using data frame reader API

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config



In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
source_path=f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name=f"{catalog_name}.{bronze_schema}.races"

In [0]:
source_path

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

races_schema = StructType([
  StructField('season', IntegerType(), True),
  StructField('round', IntegerType(), True),
  StructField('url', StringType(), True),
  StructField('raceName', StringType(), True),
  StructField('date', DateType(), True),
  StructField('circuitId', StringType(), True)  
])

df_races = (spark.read.format("csv")
               .option('header', True)
             #  .option('mode', 'FAILFAST')  # strict mode datatype validation
             #  .option('mod', 'PERMISSIVE') # ignore bad records with null value
              # .option('inferSchema', True)  optional incase of schema passing as below
               .schema(races_schema)
               .load(source_path))

In [0]:
df_races.show();

In [0]:
import pyspark.sql.functions as F

df_races_final=add_ingestion_metadata(df_races)
display(df_races_final)

In [0]:
write_to_bronze(input_df=df_races_final,
                target_table=table_name,
                batch_id=v_batch_id)

In [0]:
# (df_races_final.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable(table_name))

In [0]:
%sql
select * from formula1_incr_catalog.bronze.races

In [0]:
df_table=spark.read.table(table_name)
display(df_table)